In [10]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(r"C:\Users\shlok\projects\ddp-llm\.env")
api_key = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=api_key)

# smoke test
resp = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Say hello in one word.",
)
print(resp.text)

Hi


In [11]:
import os, json, time, random
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(r"C:\Users\shlok\projects\ddp-llm\.env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm")
seed = [json.loads(l) for l in (PROJECT / "data" / "seed.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"loaded {len(seed)} seed examples")

loaded 50 seed examples


In [12]:
def categorize(ex):
    l = ex["label"]
    u = ex["utterance"].lower()
    if l == "*":
        uncertain_markers = ["don't know", "no idea", "can't tell", "same to me", "not sure"]
        if any(m in u for m in uncertain_markers):
            return "uncertainty"
        return "off_topic"
    if l == []:
        return "reject_all"
    if isinstance(l, list):
        neg_markers = ["not ", "anything but", "except", "rule out", "wrong", "eliminate"]
        if any(m in u for m in neg_markers):
            return "negation"
        comp_markers = [" better ", " beats ", " prefer ", " over ", " more than "]
        if any(m in u for m in comp_markers):
            return "comparative"
        if len(l) == 1:
            return "single_positive"
        return "multi_positive"
    return "other"

from collections import Counter, defaultdict
by_cat = defaultdict(list)
for ex in seed:
    by_cat[categorize(ex)].append(ex)

for cat, items in by_cat.items():
    print(f"{cat}: {len(items)}")

single_positive: 10
multi_positive: 15
negation: 8
comparative: 4
reject_all: 5
off_topic: 4
uncertainty: 4


In [13]:
TASK_SPEC = """The user is shown 4 options labeled A, B, C, D and gives natural-language feedback about which are close to what they want. A parser converts that feedback into a subset of the options.

Output format for a training example is JSON with fields:
- utterance: a natural language string
- label: one of
  - a JSON list of favored option letters, e.g. ["A", "B"]
  - [] if the user explicitly rejects ALL options
  - "*" if the utterance is off-topic OR expresses no usable preference

Labeling rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- "I don't know" / "they all look the same" -> "*" (no info gain).
- "none of these" / "all wrong" -> [] (explicit rejection).
"""

In [14]:
CATEGORY_PLAN = {
    "single_positive":  {"n": 150, "desc": "The user expresses positive sentiment about exactly one option. Label is a single-element list."},
    "multi_positive":   {"n": 200, "desc": "The user expresses positive sentiment about 2, 3, or 4 options without using negation or comparatives. Label is a list with 2+ elements."},
    "comparative":      {"n": 300, "desc": "The user compares options using words like 'better than', 'prefer X over Y', 'X beats Y', 'more than'. Mix of two subcases: (a) 'X is better than Y' with no other endorsement -> only [X]; (b) 'X and Y are both good, X better' -> [X, Y]. Include both subcases roughly evenly."},
    "negation":         {"n": 250, "desc": "The user uses negation or elimination language: 'not D', 'anything but B', 'rule out A', 'C is wrong', 'except for', 'eliminate'. Label is the set of REMAINING options after eliminating the negated ones."},
    "reject_all":       {"n": 200, "desc": "The user explicitly rejects all 4 options. Phrases like 'none of these', 'all wrong', 'nothing works', 'I hate all of them'. Label is []."},
    "off_topic":        {"n": 150, "desc": "The user says something off-topic or irrelevant to the task. Random remarks, questions unrelated to the options, filler. Label is \"*\"."},
    "uncertainty":      {"n": 150, "desc": "The user expresses uncertainty or inability to distinguish: 'I don't know', 'they all look the same', 'no idea', 'can't tell', 'unsure'. Label is \"*\"."},
    "mixed_sentiment":  {"n": 200, "desc": "The user gives mixed per-option verdicts, some positive some negative, e.g. 'A great, B meh, C terrible, D good' -> ['A','D']. Each option is judged individually."},
}
print(sum(c["n"] for c in CATEGORY_PLAN.values()), "total examples planned")

1600 total examples planned


In [17]:
def make_prompt(category, category_desc, demos, n_examples):
    demo_str = "\n".join(json.dumps({"utterance": d["utterance"], "label": d["label"]}, ensure_ascii=False) for d in demos)
    return f"""{TASK_SPEC}

You are generating {n_examples} training examples for ONE specific category: "{category}".

Category description: {category_desc}

Here are example demonstrations of this exact category:
{demo_str}

Now generate {n_examples} NEW examples in this category. Requirements:
- Do NOT copy or lightly paraphrase the demonstrations. Every utterance must be genuinely new.
- Vary aggressively: length (from 1 word to a full rambling sentence), punctuation, capitalization, register (formal, casual, terse, slang), typos, sentence structure (statements, imperatives, questions where appropriate).
- Labels MUST strictly follow the rules for this category.
- Options are always ["A", "B", "C", "D"].

Return ONLY a JSON array of exactly {n_examples} objects with fields "utterance" and "label". No prose, no explanation."""

def generate_batch(category, n_examples=30):
    plan = CATEGORY_PLAN[category]
    seed_cat = by_cat.get(category, [])
    # If this category has no seed, use mixed_sentiment/multi_positive as fallback for shape
    if not seed_cat:
        seed_cat = by_cat.get("multi_positive", [])
    demos = random.sample(seed_cat, min(5, len(seed_cat)))
    prompt = make_prompt(category, plan["desc"], demos, n_examples)
    
    resp = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=1.0,
        ),
    )
    return resp.text

In [18]:
random.seed(0)
raw = generate_batch("negation", n_examples=10)
print(raw[:2000])

[
  {
    "utterance": "Rule out D, C looks best",
    "label": ["A", "B", "C"]
  },
  {
    "utterance": "I can't see myself picking A or B, C and D are better",
    "label": ["C", "D"]
  },
  {
    "utterance": "just not B",
    "label": ["A", "C", "D"]
  },
  {
    "utterance": "A is out, so is C.",
    "label": ["B", "D"]
  },
  {
    "utterance": "Anything other than D, please.",
    "label": ["A", "B", "C"]
  },
  {
    "utterance": "none of these are good except maybe C",
    "label": ["C"]
  },
  {
    "utterance": "B is definitely not it. How about the others?",
    "label": ["A", "C", "D"]
  },
  {
    "utterance": "I'm rejecting A, B, and C.",
    "label": ["D"]
  },
  {
    "utterance": "Eliminate A. The rest are okay.",
    "label": ["B", "C", "D"]
  },
  {
    "utterance": "Not A, not B, not C. What's left?",
    "label": ["D"]
  }
]


In [19]:
from tqdm import tqdm

BATCH_SIZE = 30  # examples per API call
SLEEP_SEC = 7    # stay under 10 RPM with margin

def validate(item):
    if not isinstance(item, dict): return False
    if "utterance" not in item or "label" not in item: return False
    if not isinstance(item["utterance"], str) or not item["utterance"].strip(): return False
    l = item["label"]
    if l == "*": return True
    if isinstance(l, list) and all(x in ["A","B","C","D"] for x in l):
        if len(set(l)) != len(l): return False  # no duplicates
        return True
    return False

def parse_response(raw):
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return [], [raw]
    if not isinstance(data, list):
        return [], [raw]
    good, bad = [], []
    for item in data:
        if validate(item):
            good.append(item)
        else:
            bad.append(item)
    return good, bad

output_path = PROJECT / "data" / "synthetic_raw.jsonl"
rejects_path = PROJECT / "data" / "synthetic_rejects.jsonl"
output_path.parent.mkdir(exist_ok=True)

collected = {cat: [] for cat in CATEGORY_PLAN}
rejects = []

random.seed(42)
for cat, plan in CATEGORY_PLAN.items():
    n_needed = plan["n"]
    n_batches = (n_needed + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"\n=== {cat}: need {n_needed}, {n_batches} batches ===")
    pbar = tqdm(range(n_batches))
    for _ in pbar:
        try:
            raw = generate_batch(cat, n_examples=BATCH_SIZE)
            good, bad = parse_response(raw)
            for g in good:
                g["options"] = ["A","B","C","D"]
                g["category"] = cat
            collected[cat].extend(good)
            for b in bad:
                rejects.append({"category": cat, "item": b})
            pbar.set_description(f"{cat} got {len(collected[cat])}/{n_needed}")
        except Exception as e:
            print(f"  batch failed: {e}")
        time.sleep(SLEEP_SEC)
        if len(collected[cat]) >= n_needed:
            break

# write out
with output_path.open("w", encoding="utf-8") as f:
    for cat, items in collected.items():
        for it in items[:CATEGORY_PLAN[cat]["n"]]:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")

with rejects_path.open("w", encoding="utf-8") as f:
    for r in rejects:
        f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")

total = sum(min(len(v), CATEGORY_PLAN[k]["n"]) for k, v in collected.items())
print(f"\nDONE. wrote {total} examples to {output_path}")
print(f"rejects: {len(rejects)} to {rejects_path}")
for cat, items in collected.items():
    print(f"  {cat}: {len(items)} collected (target {CATEGORY_PLAN[cat]['n']})")


=== single_positive: need 150, 5 batches ===


single_positive got 60/150:  40%|████      | 2/5 [00:31<00:47, 15.75s/it]

  batch failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


single_positive got 60/150:  60%|██████    | 3/5 [00:44<00:29, 14.61s/it]

  batch failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


...

  batch failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 5.402044879s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-f

 57%|█████▋    | 4/7 [00:29<00:21,  7.30s/it]

  batch failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 58.062032544s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-

 57%|█████▋    | 4/7 [00:36<00:27,  9.10s/it]


KeyboardInterrupt: 

In [20]:
for cat, items in collected.items():
    have = len(items)
    need = CATEGORY_PLAN[cat]["n"]
    marker = "✓" if have >= need else "…"
    print(f"  {marker} {cat}: {have}/{need}")
print(f"total: {sum(len(v) for v in collected.values())}")

  … single_positive: 90/150
  … multi_positive: 120/200
  … comparative: 120/300
  … negation: 0/250
  … reject_all: 0/200
  … off_topic: 0/150
  … uncertainty: 0/150
  … mixed_sentiment: 0/200
total: 330


In [21]:
from pathlib import Path
import json
from collections import Counter

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm")
path = PROJECT / "data" / "synthetic_raw.jsonl"
examples = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]

print(f"total examples: {len(examples)}")
cat_counts = Counter(e.get("category", "unknown") for e in examples)
for cat, n in sorted(cat_counts.items()):
    print(f"  {cat}: {n}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\shlok\\projects\\ddp-llm\\data\\synthetic_raw.jsonl'

In [22]:
import os
data_dir = r"C:\Users\shlok\projects\ddp-llm\data"
for f in os.listdir(data_dir):
    print(f, os.path.getsize(os.path.join(data_dir, f)))

seed.jsonl 4684


In [1]:
from pathlib import Path
import json
from collections import Counter

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm")
path = PROJECT / "data" / "synthetic_raw.jsonl"
examples = []
errors = []
for i, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
    line = line.strip()
    if not line:
        continue
    try:
        examples.append(json.loads(line))
    except json.JSONDecodeError as e:
        errors.append((i, line[:80], str(e)))

print(f"total valid examples: {len(examples)}")
print(f"parse errors: {len(errors)}")
for i, snippet, err in errors[:5]:
    print(f"  line {i}: {snippet!r} -> {err}")

cat_counts = Counter(e.get("category", "unknown") for e in examples)
for cat, n in sorted(cat_counts.items()):
    print(f"  {cat}: {n}")

total valid examples: 1356
parse errors: 0
  comparative: 180
  mixed_sentiment: 200
  multi_positive: 80
  negation: 286
  off_topic: 150
  reject_all: 200
  single_positive: 60
  uncertainty: 150
  unknown: 50


In [2]:
import json, random
from pathlib import Path
from collections import Counter, defaultdict

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm")
seed_path = PROJECT / "data" / "seed.jsonl"
synth_path = PROJECT / "data" / "synthetic_raw.jsonl"

seed = [json.loads(l) for l in seed_path.read_text(encoding="utf-8").splitlines() if l.strip()]
synth = [json.loads(l) for l in synth_path.read_text(encoding="utf-8").splitlines() if l.strip()]

# Tag seed with a category for tracking (not needed for training, just bookkeeping)
def categorize(ex):
    l = ex["label"]
    u = ex["utterance"].lower()
    if l == "*":
        if any(m in u for m in ["don't know", "no idea", "can't tell", "same to me", "not sure", "clueless", "idk", "dunno"]):
            return "uncertainty"
        return "off_topic"
    if l == []:
        return "reject_all"
    if isinstance(l, list):
        if any(m in u for m in ["not ", "anything but", "except", "rule out", "wrong", "eliminate", "avoid", "no on", "skip", "veto"]):
            return "negation"
        if any(m in u for m in [" better ", " beats ", " prefer ", " over ", " more than ", ">", " ahead of ", " above "]):
            return "comparative"
        if len(l) == 1:
            return "single_positive"
        return "multi_positive"
    return "other"

for ex in seed:
    if "category" not in ex:
        ex["category"] = categorize(ex)

# Combine and dedupe
all_examples = seed + synth
seen = set()
unique = []
for ex in all_examples:
    key = (ex["utterance"].lower().strip(), json.dumps(ex["label"], sort_keys=True))
    if key in seen:
        continue
    seen.add(key)
    unique.append(ex)

print(f"combined: {len(all_examples)}, unique: {len(unique)}, dupes removed: {len(all_examples) - len(unique)}")

# Stratified split
by_cat = defaultdict(list)
for ex in unique:
    by_cat[ex.get("category", "other")].append(ex)

random.seed(42)
train, test = [], []
for cat, items in by_cat.items():
    random.shuffle(items)
    n_test = max(1, len(items) // 7)  # ~15%
    test.extend(items[:n_test])
    train.extend(items[n_test:])

random.shuffle(train)
random.shuffle(test)

print(f"train: {len(train)}, test: {len(test)}")
print("train per category:", Counter(e["category"] for e in train))
print("test  per category:", Counter(e["category"] for e in test))

train_path = PROJECT / "data" / "train.jsonl"
test_path = PROJECT / "data" / "test.jsonl"
with train_path.open("w", encoding="utf-8") as f:
    for ex in train:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
with test_path.open("w", encoding="utf-8") as f:
    for ex in test:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"\nsaved to {train_path} and {test_path}")

combined: 1406, unique: 1353, dupes removed: 53
train: 1162, test: 191
train per category: Counter({'negation': 252, 'reject_all': 175, 'mixed_sentiment': 171, 'comparative': 158, 'off_topic': 132, 'uncertainty': 132, 'multi_positive': 82, 'single_positive': 60})
test  per category: Counter({'negation': 42, 'reject_all': 29, 'mixed_sentiment': 28, 'comparative': 26, 'uncertainty': 22, 'off_topic': 22, 'multi_positive': 13, 'single_positive': 9})

saved to C:\Users\shlok\projects\ddp-llm\data\train.jsonl and C:\Users\shlok\projects\ddp-llm\data\test.jsonl
